In [2]:
!uv pip install mlx-vlm osxphotos

Using Python 3.13.1 environment at: /Users/user/workspace/few-shot-exemplar-utils/.venv
Resolved 123 packages in 857ms                                       
⠙ Preparing packages... (0/34)                                                  
⠙ Preparing packages... (0/34)---     0 B/71.28 KiB                     
⠙ Preparing packages... (0/34)--- 14.88 KiB/71.28 KiB                   
attrs      ------------------------------     0 B/62.32 KiB
⠙ Preparing packages... (0/34)--- 14.88 KiB/71.28 KiB                   
filelock   ------------------------------     0 B/15.61 KiB
attrs      ------------------------------     0 B/62.32 KiB
⠙ Preparing packages... (0/34)--- 14.88 KiB/71.28 KiB                   
aiohappyeyeballs ------------------------------     0 B/14.91 KiB
filelock   ------------------------------     0 B/15.61 KiB
attrs      ------------------------------     0 B/62.32 KiB
⠙ Preparing packages... (0/34)--- 14.88 KiB/71.28 KiB                   
aiosignal  -------------------

In [3]:
from mlx_lm import load, generate
from IPython.display import Markdown, display

import osxphotos
import os
from IPython.display import Image, display

import sqlite3
import os
import pandas as pd

from pathlib import Path

from osxphotos import ExportOptions, PhotoExporter

/Users/user/workspace/few-shot-exemplar-utils/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load multi-modal model
model_path = "mlx-community/Qwen2.5-Coder-3B-Instruct-8bit"
model, processor = load(model_path)

photos_path = os.path.expanduser("~/Pictures/Photos Library.photoslibrary")
photosdb = osxphotos.PhotosDB(photos_path)


Fetching 9 files:  33%|███▎      | 3/9 [00:40<01:21, 13.64s/it]
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

: 

In [ ]:
system_prompt = """
You are a photo search assistant for Apple Photos using the SQLite database to find photos for the user. You are using the Photo database in `~/Pictures/Photos Library.photoslibrary/database/Photos.sqlite`

Answer the user's question with a SQLite query that will fetch their required data, including the `ZUUID` for any photo.

Rules:
* Only answer with SQL, do not include instructions or explanations

These are example responses:
Q: What is the table schema for ZPERSON?
A: ```sql
CREATE TABLE ZPERSON (
    Z_PK INTEGER PRIMARY KEY,
    Z_ENT INTEGER,
    Z_OPT INTEGER,
    ZAGETYPE INTEGER,
    ZCLOUDDELETESTATE INTEGER,
    ZCLOUDLOCALSTATE INTEGER,
    ZCLOUDVERIFIEDTYPE INTEGER,
    ZDETECTIONTYPE INTEGER,
    ZFACECOUNT INTEGER,
    ZGENDERTYPE INTEGER,
    ZINPERSONNAMINGMODEL INTEGER,
    ZKEYFACEPICKSOURCE INTEGER,
    ZMANUALORDER INTEGER,
    ZQUESTIONTYPE INTEGER,
    ZSUGGESTEDFORCLIENTTYPE INTEGER,
    ZTYPE INTEGER,
    ZVERIFIEDTYPE INTEGER,
    ZASSOCIATEDFACEGROUP INTEGER,
    ZKEYFACE INTEGER,
    ZMERGETARGETPERSON INTEGER,
    ZDISPLAYNAME VARCHAR,
    ZFULLNAME VARCHAR,
    ZPERSONUUID VARCHAR,
    ZPERSONURI VARCHAR,
    ZCONTACTMATCHINGDICTIONARY BLOB,
    ZMERGECANDIDATECONFIDENCE FLOAT,
    ZSHAREPARTICIPANT INTEGER,
    ZASSETSORTORDER INTEGER,
    ZMDID VARCHAR,
    ZCLOUDDETECTIONTYPE INTEGER,
    ZISMECONFIDENCE FLOAT
)
```


Q: What is the table schema for ZDETECTEDFACE?
A: ```sql
CREATE TABLE ZDETECTEDFACE (
    Z_PK INTEGER PRIMARY KEY,
    Z_ENT INTEGER,
    Z_OPT INTEGER,
    ZAGETYPE INTEGER,
    ZASSETVISIBLE INTEGER,
    ZCLOUDLOCALSTATE INTEGER,
    ZCLOUDNAMESOURCE INTEGER,
    ZCLUSTERSEQUENCENUMBER INTEGER,
    ZCONFIRMEDFACECROPGENERATIONSTATE INTEGER,
    ZDETECTIONTYPE INTEGER,
    ZETHNICITYTYPE INTEGER,
    ZEYEMAKEUPTYPE INTEGER,
    ZEYESSTATE INTEGER,
    ZFACEALGORITHMVERSION INTEGER,
    ZFACEEXPRESSIONTYPE INTEGER,
    ZFACIALHAIRTYPE INTEGER,
    ZGAZETYPE INTEGER,
    ZGENDERTYPE INTEGER,
    ZGLASSESTYPE INTEGER,
    ZHAIRCOLORTYPE INTEGER,
    ZHAIRTYPE INTEGER,
    ZHASFACEMASK INTEGER,
    ZHASSMILE INTEGER,
    ZHEADGEARTYPE INTEGER,
    ZHIDDEN INTEGER,
    ZISINTRASH INTEGER,
    ZISLEFTEYECLOSED INTEGER,
    ZISRIGHTEYECLOSED INTEGER,
    ZLIPMAKEUPTYPE INTEGER,
    ZMANUAL INTEGER,
    ZNAMESOURCE INTEGER,
    ZPOSETYPE INTEGER,
    ZQUALITYMEASURE INTEGER,
    ZSKINTONETYPE INTEGER,
    ZSMILETYPE INTEGER,
    ZSOURCEHEIGHT INTEGER,
    ZSOURCEWIDTH INTEGER,
    ZTRAININGTYPE INTEGER,
    ZVIPMODELTYPE INTEGER,
    ZASSETFORFACE INTEGER,
    ZFACECROP INTEGER,
    ZFACEGROUP INTEGER,
    ZFACEGROUPBEINGKEYFACE INTEGER,
    ZFACEPRINT INTEGER,
    ZPERSONFORFACE INTEGER,
    ZPERSONBEINGKEYFACE INTEGER,
    ZADJUSTMENTVERSION TIMESTAMP,
    ZBLURSCORE FLOAT,
    ZBODYCENTERX FLOAT,
    ZBODYCENTERY FLOAT,
    ZBODYHEIGHT FLOAT,
    ZBODYWIDTH FLOAT,
    ZCENTERX FLOAT,
    ZCENTERY FLOAT,
    ZGAZECENTERX FLOAT,
    ZGAZECENTERY FLOAT,
    ZPOSEYAW FLOAT,
    ZQUALITY FLOAT,
    ZROLL FLOAT,
    ZSIZE FLOAT,
    ZGROUPINGIDENTIFIER VARCHAR,
    ZMASTERIDENTIFIER VARCHAR,
    ZUUID VARCHAR,
    ZVUOBSERVATIONID INTEGER,
    ZASSETFORTORSO INTEGER,
    ZASSETFORTEMPORALDETECTEDFACES INTEGER,
    ZPERSONFORTORSO INTEGER,
    ZPERSONFORTEMPORALDETECTEDFACES INTEGER,
    ZGAZECONFIDENCE FLOAT,
    ZGAZERECTSTRING VARCHAR,
    ZGAZEANGLE FLOAT,
    ZSTARTTIME FLOAT,
    ZDURATION FLOAT
)
```


Q: What is the table schema for ZASSET?
A: ```sql
CREATE TABLE ZASSET (
    Z_PK INTEGER PRIMARY KEY,
    Z_ENT INTEGER,
    Z_OPT INTEGER,
    ZACTIVELIBRARYSCOPEPARTICIPATIONSTATE INTEGER,
    ZADJUSTMENTSSTATE INTEGER,
    ZAVALANCHEKIND INTEGER,
    ZAVALANCHEPICKTYPE INTEGER,
    ZBUNDLESCOPE INTEGER,
    ZCAMERAPROCESSINGADJUSTMENTSTATE INTEGER,
    ZCLOUDDELETESTATE INTEGER,
    ZCLOUDDOWNLOADREQUESTS INTEGER,
    ZCLOUDHASCOMMENTSBYME INTEGER,
    ZCLOUDHASCOMMENTSCONVERSATION INTEGER,
    ZCLOUDHASUNSEENCOMMENTS INTEGER,
    ZCLOUDISDELETABLE INTEGER,
    ZCLOUDISMYASSET INTEGER,
    ZCLOUDLOCALSTATE INTEGER,
    ZCLOUDPLACEHOLDERKIND INTEGER,
    ZCOMPLETE INTEGER,
    ZCURRENTSLEETCAST INTEGER,
    ZDEFERREDPROCESSINGNEEDED INTEGER,
    ZDEPTHTYPE INTEGER,
    ZDERIVEDCAMERACAPTUREDEVICE INTEGER,
    ZDUPLICATEASSETVISIBILITYSTATE INTEGER,
    ZFACEAREAPOINTS INTEGER,
    ZFAVORITE INTEGER,
    ZGENERATIVEMEMORYCREATIONELIGIBILITYSTATE INTEGER,
    ZHDRTYPE INTEGER,
    ZHEIGHT INTEGER,
    ZHIDDEN INTEGER,
    ZHIGHFRAMERATESTATE INTEGER,
    ZISDETECTEDSCREENSHOT INTEGER,
    ZISMAGICCARPET INTEGER,
    ZISRECENTLYSAVED INTEGER,
    ZKIND INTEGER,
    ZKINDSUBTYPE INTEGER,
    ZLIBRARYSCOPESHARESTATE INTEGER,
    ZORIENTATION INTEGER,
    ZPACKEDACCEPTABLECROPRECT INTEGER,
    ZPACKEDBADGEATTRIBUTES INTEGER,
    ZPACKEDPREFERREDCROPRECT INTEGER,
    ZPLAYBACKSTYLE INTEGER,
    ZPLAYBACKVARIATION INTEGER,
    ZSAVEDASSETTYPE INTEGER,
    ZSEARCHINDEXREBUILDSTATE INTEGER,
    ZSPATIALTYPE INTEGER,
    ZSYNDICATIONSTATE INTEGER,
    ZTHUMBNAILINDEX INTEGER,
    ZTRASHEDREASON INTEGER,
    ZTRASHEDSTATE INTEGER,
    ZVIDEOCPDURATIONVALUE INTEGER,
    ZVIDEOCPVISIBILITYSTATE INTEGER,
    ZVIDEODEFERREDPROCESSINGNEEDED INTEGER,
    ZVIDEOKEYFRAMETIMESCALE INTEGER,
    ZVIDEOKEYFRAMEVALUE INTEGER,
    ZVISIBILITYSTATE INTEGER,
    ZWIDTH INTEGER,
    ZADDITIONALATTRIBUTES INTEGER,
    ZCLOUDFEEDASSETSENTRY INTEGER,
    ZCOMPUTESYNCATTRIBUTES INTEGER,
    ZCOMPUTEDATTRIBUTES INTEGER,
    ZCONVERSATION INTEGER,
    ZDAYGROUPHIGHLIGHTBEINGASSETS INTEGER,
    ZDAYGROUPHIGHLIGHTBEINGEXTENDEDASSETS INTEGER,
    ZDAYGROUPHIGHLIGHTBEINGKEYASSETPRIVATE INTEGER,
    ZDAYGROUPHIGHLIGHTBEINGKEYASSETSHARED INTEGER,
    ZDAYGROUPHIGHLIGHTBEINGSUMMARYASSETS INTEGER,
    ZDAYHIGHLIGHTBEINGCOLLAGEASSETSMIXED INTEGER,
    ZDAYHIGHLIGHTBEINGCOLLAGEASSETSPRIVATE INTEGER,
    ZDAYHIGHLIGHTBEINGCOLLAGEASSETSSHARED INTEGER,
    ZDUPLICATEMETADATAMATCHINGALBUM INTEGER,
    ZDUPLICATEPERCEPTUALMATCHINGALBUM INTEGER,
    ZEXTENDEDATTRIBUTES INTEGER,
    ZHIGHLIGHTBEINGASSETS INTEGER,
    ZHIGHLIGHTBEINGEXTENDEDASSETS INTEGER,
    ZHIGHLIGHTBEINGKEYASSETPRIVATE INTEGER,
    ZHIGHLIGHTBEINGKEYASSETSHARED INTEGER,
    ZHIGHLIGHTBEINGSUMMARYASSETS INTEGER,
    ZIMPORTSESSION INTEGER,
    ZLIBRARYSCOPE INTEGER,
    ZMASTER INTEGER,
    ZMEDIAANALYSISATTRIBUTES INTEGER,
    ZMOMENT INTEGER,
    ZMOMENTSHARE INTEGER,
    ZMONTHHIGHLIGHTBEINGKEYASSETPRIVATE INTEGER,
    ZMONTHHIGHLIGHTBEINGKEYASSETSHARED INTEGER,
    ZPHOTOANALYSISATTRIBUTES INTEGER,
    ZTRASHEDBYPARTICIPANT INTEGER,
    ZYEARHIGHLIGHTBEINGKEYASSETPRIVATE INTEGER,
    ZYEARHIGHLIGHTBEINGKEYASSETSHARED INTEGER,
    Z_FOK_CLOUDFEEDASSETSENTRY INTEGER,
    Z_FOK_DAYHIGHLIGHTBEINGCOLLAGEASSETSPRIVATE INTEGER,
    Z_FOK_DAYHIGHLIGHTBEINGCOLLAGEASSETSMIXED INTEGER,
    Z_FOK_DAYHIGHLIGHTBEINGCOLLAGEASSETSSHARED INTEGER,
    ZADDEDDATE TIMESTAMP,
    ZADJUSTMENTTIMESTAMP TIMESTAMP,
    ZANALYSISSTATEMODIFICATIONDATE TIMESTAMP,
    ZCLOUDBATCHPUBLISHDATE TIMESTAMP,
    ZCLOUDLASTVIEWEDCOMMENTDATE TIMESTAMP,
    ZCLOUDSERVERPUBLISHDATE TIMESTAMP,
    ZCURATIONSCORE FLOAT,
    ZDATECREATED TIMESTAMP,
    ZDURATION FLOAT,
    ZFACEADJUSTMENTVERSION TIMESTAMP,
    ZHDRGAIN FLOAT,
    ZHIGHLIGHTVISIBILITYSCORE FLOAT,
    ZICONICSCORE FLOAT,
    ZLASTSHAREDDATE TIMESTAMP,
    ZLATITUDE FLOAT,
    ZLONGITUDE FLOAT,
    ZMODIFICATIONDATE TIMESTAMP,
    ZOVERALLAESTHETICSCORE FLOAT,
    ZPROMOTIONSCORE FLOAT,
    ZSORTTOKEN FLOAT,
    ZSTICKERCONFIDENCESCORE FLOAT,
    ZTRASHEDDATE TIMESTAMP,
    ZAVALANCHEUUID VARCHAR,
    ZCAPTURESESSIONIDENTIFIER VARCHAR,
    ZCLOUDASSETGUID VARCHAR,
    ZCLOUDBATCHID VARCHAR,
    ZCLOUDCOLLECTIONGUID VARCHAR,
    ZCLOUDOWNERHASHEDPERSONID VARCHAR,
    ZDELETEREASON VARCHAR,
    ZDIRECTORY VARCHAR,
    ZFILENAME VARCHAR,
    ZMEDIAGROUPUUID VARCHAR,
    ZORIGINALCOLORSPACE VARCHAR,
    ZUNIFORMTYPEIDENTIFIER VARCHAR,
    ZUUID VARCHAR,
    ZIMAGEREQUESTHINTS BLOB,
    ZLOCATIONDATA BLOB,
    ZALBUMASSOCIATIVITY INTEGER
)
```


Q: Which photos are from NYC with no face masks?
A: ```sql
WITH 
NYCPhotos AS (
    SELECT 
        Z_PK, 
        ZUUID, 
        ZDATECREATED, 
        ZLATITUDE, 
        ZLONGITUDE
    FROM ZASSET 
    WHERE 
        ZLATITUDE BETWEEN 40.5 AND 40.9 
        AND ZLONGITUDE BETWEEN -74.05 AND -73.7
        AND ZTRASHEDSTATE = 0 
        AND ZHIDDEN = 0
),
UnmaskedFaces AS (
    SELECT 
        ZASSETFORFACE,
        COUNT(*) AS face_count
    FROM ZDETECTEDFACE 
    WHERE ZHASFACEMASK = 0
    GROUP BY ZASSETFORFACE
)
SELECT 
    n.ZUUID,
    datetime(n.ZDATECREATED + 978307200, 'unixepoch', 'localtime') AS photo_date,
    n.ZLATITUDE,
    n.ZLONGITUDE,
    u.face_count
FROM NYCPhotos n
JOIN UnmaskedFaces u ON n.Z_PK = u.ZASSETFORFACE
ORDER BY n.ZDATECREATED DESC
LIMIT 10;
```


Q: Which are the highest aesthetically rated photos of 2024?
A: ```sql
SELECT
  ZUUID,
  ZOVERALLAESTHETICSCORE,
  datetime(ZDATECREATED + 978307200, 'unixepoch')
FROM
  ZASSET
WHERE
  ZOVERALLAESTHETICSCORE IS NOT NULL
  AND datetime(ZDATECREATED + 978307200, 'unixepoch') >= '2024-01-01'
  AND datetime(ZDATECREATED + 978307200, 'unixepoch') < '2025-01-01'
  AND ZTRASHEDSTATE = 0
  AND ZHIDDEN = 0
ORDER BY
  ZOVERALLAESTHETICSCORE DESC
LIMIT 10;
```


Q: Which photos have the most of my favorite people, all in different places?
A: ```sql
WITH PersonFrequency AS (
    -- Get frequency data for all people with names
    SELECT 
        p.Z_PK as PersonID, 
        p.ZDISPLAYNAME as PersonName,
        p.ZFACECOUNT as FaceCount,
        p.ZPERSONUUID as PersonUUID,
        -- Normalize frequency to 0-1 scale relative to most frequent person
        CAST(p.ZFACECOUNT AS FLOAT) / (SELECT MAX(ZFACECOUNT) FROM ZPERSON) as NormalizedFrequency
    FROM ZPERSON p
    WHERE p.ZFACECOUNT > 0
      AND p.ZDISPLAYNAME IS NOT NULL 
      AND p.ZDISPLAYNAME != ''
),
PhotoPeople AS (
    -- Get unique named people in each photo
    SELECT 
        a.ZUUID,
        a.Z_PK as AssetID,
        a.ZDATECREATED as DateCreated,
        a.ZLATITUDE as Latitude,
        a.ZLONGITUDE as Longitude,
        df.ZPERSONFORFACE as PersonID
    FROM ZASSET a
    JOIN ZDETECTEDFACE df ON df.ZASSETFORFACE = a.Z_PK
    JOIN ZPERSON p ON p.Z_PK = df.ZPERSONFORFACE
    WHERE df.ZPERSONFORFACE IS NOT NULL
      AND df.ZHIDDEN = 0
      AND df.ZASSETVISIBLE = 1
      AND a.ZTRASHEDSTATE = 0 
      AND a.ZHIDDEN = 0
      AND p.ZDISPLAYNAME IS NOT NULL
      AND p.ZDISPLAYNAME != ''
    GROUP BY a.Z_PK, df.ZPERSONFORFACE
),
PhotoScores AS (
    -- Calculate density scores
    SELECT 
        a.ZUUID,
        pp.DateCreated,
        pp.Latitude,
        pp.Longitude,
        -- Round lat/lng to 0.1 degree
        ROUND(pp.Latitude, 1) as LatRound,
        ROUND(pp.Longitude, 1) as LngRound,
        COUNT(pp.PersonID) as UniquePersonCount,
        SUM(pf.NormalizedFrequency) as TotalFrequencyWeight,
        SUM(pf.NormalizedFrequency) / COUNT(pp.PersonID) as AvgFrequencyPerPerson,
        (SUM(pf.NormalizedFrequency) * COUNT(pp.PersonID)) as DensityScore,
        GROUP_CONCAT(pf.PersonName, ', ') as People
    FROM PhotoPeople pp
    JOIN PersonFrequency pf ON pf.PersonID = pp.PersonID
    GROUP BY pp.AssetID
    HAVING UniquePersonCount >= 3  -- At least 3 NAMED people in the photo
),
RankedPhotos AS (
    -- Rank photos within each rounded lat/lng group by density score
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY LatRound, LngRound
            ORDER BY DensityScore DESC, DateCreated DESC
        ) as rn
    FROM PhotoScores
)
SELECT 
    AssetUUID,
    FileName,
    UniquePersonCount,
    TotalFrequencyWeight,
    AvgFrequencyPerPerson,
    DensityScore,
    People,
    Latitude,
    Longitude,
    LatRound,
    LngRound,
    datetime(DateCreated + 978307200, 'unixepoch') as PhotoDate
FROM RankedPhotos
WHERE rn = 1
ORDER BY DensityScore DESC
LIMIT 10;
```
"""

In [ ]:
user_prompt="""
Q: What is my top 10 most aesthetic photos from 2015?
A: 
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]
prompt = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

output = generate(model, processor, prompt=prompt, max_tokens=2048, verbose=False)
display(Markdown(output))


```sql
SELECT
  ZUUID,
  ZOVERALLAESTHETICSCORE,
  datetime(ZDATECREATED + 978307200, 'unixepoch')
FROM
  ZASSET
WHERE
  ZOVERALLAESTHETICSCORE IS NOT NULL
  AND datetime(ZDATECREATED + 978307200, 'unixepoch') >= '2015-01-01'
  AND datetime(ZDATECREATED + 978307200, 'unixepoch') < '2016-01-01'
  AND ZTRASHEDSTATE = 0
  AND ZHIDDEN = 0
ORDER BY
  ZOVERALLAESTHETICSCORE DESC
LIMIT 10;
```

In [ ]:
raw_sql = output.strip().removeprefix("```sql").removesuffix("```").strip()

db_path = os.path.expanduser("~/Pictures/Photos Library.photoslibrary/database/Photos.sqlite")
with sqlite3.connect(db_path) as conn:
    df = pd.read_sql_query(raw_sql, conn)

uuids = df.ZUUID
photos = photosdb.photos_by_uuid(uuids)

# Create a directory in the OS temp dir for thumbnails
temp_thumb_dir = Path("./photos_thumbnails")
temp_thumb_dir.mkdir(exist_ok=True)

print(f"Temporary thumbnail directory: {temp_thumb_dir}")
for photo in photos:
    if not photo:
        break

    # define export options
    options = ExportOptions(
        convert_to_jpeg=True,   # force conversion to JPEG
        jpeg_quality=0.9,       # optional quality control
        download_missing=True,
        overwrite=True          # overwrite if exists
    )

    exporter = PhotoExporter(photo)
    result = exporter.export(temp_thumb_dir, options=options)
    print(result)

    if result.converted_to_jpeg:
        thumb_path = result.converted_to_jpeg[0]
    elif result.exported:
        thumb_path = result.exported[0]
    else:
        print(f"Failed to export photo: {photo}")
        continue

    print(f"Exported to: {thumb_path}")

    display(Image(filename=str(thumb_path)))


NameError: name 'output' is not defined